# SICOR • Cerrado 2021–2025
## Gráficos para complementar a seção 5.2.1

**Como usar:** abra este arquivo no Google Colab; execute as células em ordem ou use
**Ambiente de execução → Executar tudo**. Autorize a montagem do seu Drive.
O caminho da Base04 já está preenchido na célula 2. Se estiver diferente no seu Drive,
altere apenas esse caminho. Não é necessário carregar shapefiles nem instalar bibliotecas.

**Saídas:** gráficos PNG (300 dpi) e PDF, tabelas CSV com separador `;`, relatório de leitura,
auditorias e um ZIP. Cada execução cria uma subpasta nova ao lado da Base04.

### O que será contado
| Unidade | Regra |
|---|---|
| Registros territoriais | Todas as linhas da base selecionada; nenhuma linha é eliminada. |
| Operações | `REF_BACEN` distintos em cada grupo. |
| Ordens/destinações | Pares distintos `REF_BACEN + NU_ORDEM`. |
| Geometrias identificadas | `id_gleba2` distintos, quando o campo existe. Não equivalem a imóveis. |
| Valores | `VL_PARC_CREDITO` uma vez por ordem, após verificar consistência entre suas linhas. |

**Contar identificadores distintos para um indicador não elimina linhas da base territorial.**
Os valores referem-se às ordens com vínculo territorial no arquivo lido: não são automaticamente
o valor integral de todas as ordens de cada contrato nem os desembolsos realizados.

Gráficos por ano e categorias são barras/colunas; histogramas representam a distribuição dos valores.
O notebook mantém essa diferença explícita nos títulos e nas tabelas.

**Limites:** não confirma quitação ou cancelamento. Diagnostica a presença dos campos necessários,
mas o cruzamento com liberações e saldos é uma análise complementar. Não reproduz como resultados
calculados os 401.294/26.345 da Tabela 3: ordens sem geometria não estão na Base04 territorial.
Os 454.817 também não são assumidos como quantidade de `ID_GLEBA` ou de `id_gleba2`.


### 1. Conectar o Google Drive
Se a pasta estiver apenas em “Compartilhados comigo”, adicione um atalho em “Meu Drive”
e ajuste o caminho na próxima célula. O notebook lê os arquivos existentes e grava somente
seus novos resultados.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


### 2. Configuração
`RECUPERAR_FINALIDADE=True` habilita a leitura, em blocos, dos arquivos anuais da pasta `Base`.
Essa etapa pode demorar alguns minutos. É usada apenas se a finalidade não estiver na Base04.
A tabela `Empreendimento.csv` será procurada nessa pasta; se ausente, tenta-se baixá-la do BCB.
Se faltarem dados, os outros gráficos continuam e a pendência fica registrada, sem inventar categorias.

A classificação de produtor é a existente em `TIPO_PRODUTOR_ANALISE`; ela não identifica pessoas únicas.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, json, re, unicodedata, hashlib, shutil, textwrap, urllib.request, codecs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display

PASTA_ENVIO = Path('/content/drive/MyDrive/Colab Notebooks/fluxo/Envio')
ARQUIVO_BASE = PASTA_ENVIO / 'Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv'
PASTA_OPERACOES = PASTA_ENVIO / 'Base'
ARQUIVO_EMPREENDIMENTOS = PASTA_OPERACOES / 'Empreendimento.csv'
RECUPERAR_FINALIDADE = True
BAIXAR_DOMINIO_BCB = True
ANOS = list(range(2021, 2026))
BLOCO = 100_000
MOSTRAR_GRAFICOS = True
BAIXAR_ZIP_AO_FINAL = False  # True: abre também o download no navegador.

# Se a detecção escolher um campo inadequado, informe aqui o nome original exato.
# Exemplo: COLUNAS_MANUAIS = {'uf':'UF_OPERACAO', 'ordem':'NU_ORDEM_GLEBA'}
COLUNAS_MANUAIS = {}

# Auto: ponto decimal ou vírgula decimal (incluindo 1.234,56).
# Um único ponto, sem vírgula, é interpretado como decimal, conforme exportação numérica.
FORMATO_NUMERICO = 'auto'  # 'auto', 'br' (milhar . e decimal ,) ou 'internacional'
if not ARQUIVO_BASE.is_file():
    raise FileNotFoundError(f'Base não encontrada. Corrija ARQUIVO_BASE na célula 2: {ARQUIVO_BASE}')
EXECUCAO = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f_UTC')
SAIDA = ARQUIVO_BASE.parent / 'graficos_secao_5_2_1' / EXECUCAO
for pasta in ['graficos', 'tabelas', 'auditoria']:
    (SAIDA / pasta).mkdir(parents=True, exist_ok=True)
AVISOS, FONTES, INVENTARIO = [], [], []
def avisar(texto):
    AVISOS.append(texto)
    print('AVISO:', texto)
print('Base:', ARQUIVO_BASE)
print('Saída:', SAIDA)
print('pandas:', pd.__version__)


### 3. Leitura e identificação dos campos
As geometrias WKT não são carregadas em memória; os indicadores usam seus identificadores.
O CSV é lido como texto para preservar referências e ordens. Não há descarte silencioso de linhas
malformadas. A inspeção de qualidade será exportada junto com os gráficos.


In [ ]:
def normalizar(nome):
    s = unicodedata.normalize('NFKD', str(nome)).encode('ascii', 'ignore').decode()
    return re.sub(r'[^A-Z0-9]+', '_', s.upper()).strip('_')

def texto(serie):
    s = serie.astype('string').str.strip()
    return s.mask(s.str.lower().isin(['', 'nan', 'none', 'null', '<na>', 'nat']))

def identificador(serie):
    # Retira somente sufixo decimal artificial. Não remove zeros iniciais.
    return texto(serie).str.replace(r'\.0+$', '', regex=True)

def numero(serie):
    s = texto(serie).str.replace('R$', '', regex=False).str.replace(r'\s+', '', regex=True)
    if FORMATO_NUMERICO == 'br':
        s = s.str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
    elif FORMATO_NUMERICO == 'internacional':
        s = s.str.replace(',', '', regex=False)
    else:
        comma = s.str.contains(',', regex=False, na=False)
        both = comma & s.str.contains('.', regex=False, na=False)
        br = both & (s.str.rfind(',') > s.str.rfind('.'))
        us = both & ~br
        s.loc[br] = s.loc[br].str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
        s.loc[us] = s.loc[us].str.replace(',', '', regex=False)
        only = comma & ~both
        s.loc[only] = s.loc[only].str.replace(',', '.', regex=False)
    return pd.to_numeric(s, errors='coerce').astype('Float64')

def datas(serie):
    s = texto(serie).str.slice(0, 10)
    iso = s.str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
    out = pd.Series(pd.NaT, index=s.index, dtype='datetime64[ns]')
    out.loc[iso] = pd.to_datetime(s.loc[iso], format='%Y-%m-%d', errors='coerce')
    out.loc[~iso] = pd.to_datetime(s.loc[~iso], format='%d/%m/%Y', errors='coerce')
    return out

def layout_csv(caminho):
    with open(caminho, 'rb') as f:
        amostra = f.read(512_000)
    for enc in ['utf-8-sig', 'cp1252', 'latin-1']:
        try:
            amostra_texto = codecs.getincrementaldecoder(enc)().decode(amostra, final=False)
            cab = amostra_texto.splitlines()[0]
            break
        except UnicodeDecodeError:
            continue
    sep = max([';', '\t', ',', '|'], key=cab.count)
    head = pd.read_csv(caminho, sep=sep, encoding=enc, nrows=0)
    return {'sep':sep, 'encoding':enc}, list(head.columns)

def localizar(colunas, candidatos):
    mapa = {normalizar(c):c for c in colunas}
    return next((mapa[normalizar(c)] for c in candidatos if normalizar(c) in mapa), None)

CANDIDATOS = {
    'ref':['REF_BACEN'],
    'ordem':['NU_ORDEM_GLEBA','NU_ORDEM','NU_ORDEM_OPERACAO'],
    'ordem_operacao':['NU_ORDEM_OPERACAO'],
    'id_registro':['ID_GLEBA'], 'geometria':['ID_GLEBA2'],
    'data':['DT_EMISSAO'], 'ano':['ANO_OPERACAO','ANO_BASE','ANO_ARQUIVO_OPERACAO'],
    'ano_territorial':['ANO_BASE'],
    'uf':['UF_OPERACAO','UF'],
    'categoria':['TIPO_PRODUTOR_ANALISE'],
    'emitente':['DESC_CATEG_EMITENTE'],
    'programa':['DESC_PROGRAMA'],
    'valor':['VL_PARC_CREDITO'],
    'finalidade':['DESC_FINALIDADE','FINALIDADE','NOME_FINALIDADE'],
    'empreendimento':['CD_EMPREENDIMENTO'],
}
config_csv, cabecalho = layout_csv(ARQUIVO_BASE)
COL = {papel: COLUNAS_MANUAIS.get(papel) or localizar(cabecalho, candidatos)
       for papel, candidatos in CANDIDATOS.items()}
for papel, c in COL.items():
    if c and c not in cabecalho:
        raise ValueError(f'Campo manual inexistente: {papel} = {c}')
for papel in ['ref','ordem']:
    if not COL[papel]:
        raise ValueError(f'Campo obrigatório não encontrado: {papel}. Colunas: {cabecalho}')
campos_financeiros = [c for c in cabecalho if any(t in normalizar(c) for t in
    ['LIBERAC','LIBERADO','SITUACAO_OPERACAO','LIQUID','CANCEL','DT_DESC','VL_DESC','MOTIVO_DESC'])]
selecionadas = sorted(set(c for c in COL.values() if c) | set(campos_financeiros))
base = pd.read_csv(ARQUIVO_BASE, usecols=selecionadas, dtype='string', **config_csv)
N_ORIGINAL = len(base)
FONTES.append({'tipo':'base_territorial','arquivo':str(ARQUIVO_BASE),
               'bytes':ARQUIVO_BASE.stat().st_size, 'linhas':N_ORIGINAL, **config_csv})
mapa_colunas = pd.DataFrame([{'PAPEL':k,'CAMPO':v or 'AUSENTE'} for k,v in COL.items()])
mapa_colunas.to_csv(SAIDA/'auditoria/campos_identificados.csv', sep=';', index=False, encoding='utf-8-sig')
display(mapa_colunas)
print(f'{N_ORIGINAL:,} registros territoriais lidos; nenhuma linha removida.')


,PAPEL,CAMPO
0,ref,REF_BACEN
1,ordem,NU_ORDEM_GLEBA
2,ordem_operacao,NU_ORDEM_OPERACAO
3,id_registro,ID_GLEBA
4,geometria,ID_GLEBA2
5,data,DT_EMISSAO
6,ano,ANO_OPERACAO
7,ano_territorial,ANO_BASE
8,uf,UF_OPERACAO
9,categoria,TIPO_PRODUTOR_ANALISE


480,679 registros territoriais lidos; nenhuma linha removida.


### 4. Preparar chaves e dimensões
O ano principal vem de `DT_EMISSAO`; quando a data falta, utiliza-se o campo de ano disponível,
com a origem registrada. Datas fora de 2021–2025 e datas ausentes continuam na base e aparecem
em grupos próprios. Também será gerada uma tabela pelo ano do arquivo territorial (`ANO_BASE`).
A UF é a informação administrativa `UF_OPERACAO`, sem novo cruzamento geográfico.


In [ ]:
def obter(papel):
    return base[COL[papel]] if COL.get(papel) else pd.Series(pd.NA, index=base.index, dtype='string')

def preparar(base):
    d = pd.DataFrame(index=base.index)
    d['_REF'] = identificador(obter('ref'))
    d['_ORDEM_NUM'] = identificador(obter('ordem'))
    d['_CHAVE'] = d['_REF'] + '|' + d['_ORDEM_NUM']
    d['_DATA'] = datas(obter('data'))
    ano_aux = pd.to_numeric(obter('ano'), errors='coerce')
    ano_aux = ano_aux.where(ano_aux.mod(1).eq(0) & ano_aux.between(1900, 2100)).astype('Int64')
    ano = d['_DATA'].dt.year.astype('Int64').fillna(ano_aux)
    d['_ANO'] = ano.astype('string').fillna('Sem ano')
    d.loc[ano.notna() & ~ano.isin(ANOS), '_ANO'] = 'Fora de 2021–2025'
    d['_FONTE_ANO'] = np.where(d['_DATA'].notna(), 'DT_EMISSAO',
                        np.where(ano_aux.notna(), COL.get('ano') or 'Ano auxiliar', 'Ausente'))
    d['_ANO_TERRITORIAL'] = texto(obter('ano_territorial')).fillna('Não informado')
    d['_UF'] = texto(obter('uf')).str.upper().fillna('Não informada')
    ufs = set('AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR PE PI RJ RN RS RO RR SC SP SE TO'.split())
    invalida = ~d['_UF'].isin(ufs | {'Não informada'})
    d.loc[invalida, '_UF'] = 'UF não reconhecida'
    d['_CATEGORIA'] = texto(obter('categoria')).fillna('Não informada')
    d['_EMITENTE'] = texto(obter('emitente')).fillna('Não informado')
    d['_FINALIDADE'] = texto(obter('finalidade'))
    d['_VALOR'] = numero(obter('valor'))
    d['_ID_REGISTRO'] = identificador(obter('id_registro'))
    d['_GEOMETRIA'] = identificador(obter('geometria'))
    return d

dados = preparar(base)
if COL['ordem_operacao']:
    operacional = identificador(obter('ordem_operacao'))
    diferentes = (dados['_ORDEM_NUM'].notna() & operacional.notna() &
                  dados['_ORDEM_NUM'].ne(operacional)).fillna(False)
    if diferentes.any():
        base.loc[diferentes].to_csv(SAIDA/'auditoria/divergencias_ordem.csv', sep=';', index=False, encoding='utf-8-sig')
        raise ValueError('Há divergências entre NU_ORDEM_GLEBA e NU_ORDEM_OPERACAO. Verifique a auditoria antes de continuar.')
if dados['_ID_REGISTRO'].dropna().duplicated().any():
    avisar('ID_GLEBA se repete. Verifique se foi fornecida uma tabela expandida de interseções em vez da Base04 geral.')
if not COL['valor']:
    avisar('VL_PARC_CREDITO ausente: os gráficos financeiros não serão gerados.')
if not COL['categoria']:
    avisar('TIPO_PRODUTOR_ANALISE ausente: categoria de produtor não será inferida.')
if not COL['uf']:
    avisar('UF_OPERACAO/UF ausente: não se interpretará CD_ESTADO sem sua tabela de domínio.')
print(dados.groupby('_FONTE_ANO', dropna=False).size().to_string())


_FONTE_ANO
DT_EMISSAO    480679


### 5. Recuperar a finalidade, quando necessário
O vínculo usa `REF_BACEN + NU_ORDEM` e, para a descrição, `CD_EMPREENDIMENTO`.
A tabela de domínio é usada integralmente, sem filtrar apenas os empreendimentos atualmente vigentes.
Descrições conflitantes para uma chave ficam como **Ambígua — verificar**.
O vínculo não multiplica as linhas territoriais.

Fontes: [estrutura dos empreendimentos — GeoCrédito Rural](https://geo-credito-rural.github.io/sicor/microdados/empreendimentos.html)
e [Empreendimento.csv — Banco Central](https://www.bcb.gov.br/htms/sicor/Empreendimento.csv).


In [ ]:
def resolver_unico(serie):
    v = texto(serie).dropna().unique()
    if len(v) == 0:
        return pd.NA
    return v[0] if len(v) == 1 else 'Ambígua — verificar'

def mapa_unico(tabela, chave, valor):
    g = tabela.dropna(subset=[chave]).groupby(chave)[valor]
    n = g.nunique()
    return g.first().where(n.le(1), 'Ambígua — verificar')

def ler_dominio():
    caminho = ARQUIVO_EMPREENDIMENTOS
    if not caminho.is_file():
        if not BAIXAR_DOMINIO_BCB:
            return None
        caminho = SAIDA/'auditoria/Empreendimento_BCB.csv'
        url = 'https://www.bcb.gov.br/htms/sicor/Empreendimento.csv'
        try:
            req = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=90) as resposta:
                conteudo = resposta.read()
            caminho.write_bytes(conteudo)
        except Exception as e:
            avisar(f'Não foi possível obter Empreendimento.csv: {type(e).__name__}. Coloque-o na pasta Base e execute novamente.')
            return None
    cfg, cols = layout_csv(caminho)
    cod = localizar(cols, ['CD_EMPREENDIMENTO'])
    des = localizar(cols, ['FINALIDADE','DESC_FINALIDADE'])
    if not cod or not des:
        avisar('Tabela de empreendimentos sem CD_EMPREENDIMENTO e FINALIDADE reconhecíveis.')
        return None
    t = pd.read_csv(caminho, usecols=[cod,des], dtype='string', **cfg)
    t['_COD'] = identificador(t[cod])
    t['_DESC'] = texto(t[des])
    FONTES.append({'tipo':'dominio_empreendimentos','arquivo':str(caminho),
                   'sha256':hashlib.sha256(caminho.read_bytes()).hexdigest()})
    return mapa_unico(t, '_COD', '_DESC')

def completar_finalidade(dados):
    if dados['_FINALIDADE'].notna().all() or not RECUPERAR_FINALIDADE:
        return dados
    dominio = ler_dominio()
    if COL['empreendimento'] and dominio is not None:
        direta = identificador(obter('empreendimento')).map(dominio)
        dados['_FINALIDADE'] = dados['_FINALIDADE'].fillna(direta)
    if dados['_FINALIDADE'].notna().all():
        return dados
    chaves_alvo = set(dados['_CHAVE'].dropna())
    encontrados = []
    for ano in ANOS:
        arq = PASTA_OPERACOES/f'SICOR_OPERACAO_BASICA_ESTADO_{ano}.csv'
        if not arq.is_file():
            avisar(f'Finalidade: arquivo anual não encontrado: {arq.name}')
            continue
        cfg, cols = layout_csv(arq)
        ref = localizar(cols, ['REF_BACEN'])
        ordem = localizar(cols, ['NU_ORDEM','NU_ORDEM_OPERACAO'])
        emp = localizar(cols, ['CD_EMPREENDIMENTO'])
        fin = localizar(cols, ['FINALIDADE','DESC_FINALIDADE'])
        if not ref or not ordem or (not fin and (not emp or dominio is None)):
            avisar(f'Finalidade: faltam chaves/código/descrição ou domínio para {arq.name}.')
            continue
        n_lidos, n_vinculados = 0, 0
        print('Lendo finalidade:', arq.name, flush=True)
        for b in pd.read_csv(arq, usecols=list({x for x in [ref,ordem,emp,fin] if x}),
                             dtype='string', chunksize=BLOCO, **cfg):
            n_lidos += len(b)
            chave = identificador(b[ref]) + '|' + identificador(b[ordem])
            manter = chave.isin(chaves_alvo)
            if not manter.any():
                continue
            c = b.loc[manter]
            f = texto(c[fin]) if fin else identificador(c[emp]).map(dominio)
            pedaco = pd.DataFrame({'_CHAVE':chave.loc[manter], '_FINALIDADE':f})
            encontrados.append(pedaco.drop_duplicates())
            n_vinculados += len(c)
        FONTES.append({'tipo':'operacoes_para_finalidade','arquivo':str(arq),
                       'linhas_lidas':n_lidos,'linhas_com_chave_territorial':n_vinculados})
    if encontrados:
        t = pd.concat(encontrados, ignore_index=True).drop_duplicates()
        mapa = mapa_unico(t, '_CHAVE', '_FINALIDADE')
        ambiguas = mapa[mapa.eq('Ambígua — verificar')]
        t[t['_CHAVE'].isin(ambiguas.index)].to_csv(SAIDA/'auditoria/finalidades_ambiguas.csv',
                                               sep=';', index=False, encoding='utf-8-sig')
        dados['_FINALIDADE'] = dados['_FINALIDADE'].fillna(dados['_CHAVE'].map(mapa))
    return dados

dados = completar_finalidade(dados)
dados['_FINALIDADE'] = dados['_FINALIDADE'].fillna('Não identificada')
if dados['_FINALIDADE'].eq('Não identificada').any():
    avisar(f"Finalidade não identificada em {dados['_FINALIDADE'].eq('Não identificada').sum():,} registros; confira a cobertura no relatório.")
assert len(dados) == N_ORIGINAL, 'A quantidade de registros foi alterada indevidamente.'
display(dados['_FINALIDADE'].value_counts(dropna=False).rename('REGISTROS').to_frame())


### 6. Auditoria das unidades e tabela financeira por ordem
Somente para os resumos financeiros, cria-se uma tabela auxiliar de ordens. Se o valor de uma
mesma ordem for repetido em várias glebas, entra uma vez. Valores divergentes, negativos,
infinitos, ausentes ou com falha de conversão são auditados e não entram na soma.
Nenhum valor é escolhido arbitrariamente entre valores divergentes.

O agrupamento financeiro é conservador: se uma ordem tem conflito de ano, UF ou categoria,
seu valor permanece no total elegível e aparece em **Múltiplos valores — verificar** naquela dimensão.
O histograma é por **ordem/destinação**, usando `VL_PARC_CREDITO`, e não por gleba.


In [ ]:
DIMENSOES = {'ano':('_ANO','Ano da operação'), 'uf':('_UF','UF administrativa'),
             'categoria_produtor':('_CATEGORIA','Categoria de produtor'),
             'finalidade':('_FINALIDADE','Finalidade'),
             'categoria_emitente':('_EMITENTE','Categoria do emitente')}

def salvar_tabela(t, nome, auditoria=False):
    p = SAIDA/('auditoria' if auditoria else 'tabelas')/f'{nome}.csv'
    t.to_csv(p, sep=';', decimal=',', index=False, encoding='utf-8-sig')
    return p

def dimensao_unica(s):
    v = s.dropna().unique()
    return v[0] if len(v)==1 else ('Não informada' if len(v)==0 else 'Múltiplos valores — verificar')

def consolidar_ordens(dados):
    d = dados[dados['_CHAVE'].notna()]
    g = d.groupby('_CHAVE', sort=False)
    out = g.agg(_REF=('_REF','first'), REGISTROS=('_REF','size'),
                N_VALORES=('_VALOR','nunique'), N_VALIDOS=('_VALOR','count'),
                VALOR_ORDEM_R=('_VALOR','first'))
    for campo, rotulo in DIMENSOES.values():
        # first só é utilizado depois de verificar a unicidade.
        n = g[campo].nunique(dropna=False)
        out[campo] = g[campo].first().where(n.eq(1), 'Múltiplos valores — verificar')
    ausente = out['N_VALIDOS'].ne(out['REGISTROS'])
    out['STATUS_VALOR'] = np.select(
        [out['N_VALORES'].gt(1), ausente,
         ~np.isfinite(out['VALOR_ORDEM_R'].astype(float)), out['VALOR_ORDEM_R'].lt(0).fillna(False)],
        ['Valores divergentes','Valor ausente ou inválido em alguma linha',
         'Valor não finito','Valor negativo'], default='Elegível')
    out.loc[out['STATUS_VALOR'].ne('Elegível'), 'VALOR_ORDEM_R'] = pd.NA
    return out.reset_index()

ordens = consolidar_ordens(dados)
ordens_validas = ordens[ordens['STATUS_VALOR'].eq('Elegível')].copy()
salvar_tabela(ordens[ordens['STATUS_VALOR'].ne('Elegível')], 'ordens_valores_nao_elegiveis', True)
salvar_tabela(ordens, 'base_financeira_por_ordem', True)
q = pd.DataFrame([
    ['Registros territoriais (linhas)',len(dados)],
    ['Operações REF_BACEN distintas',dados['_REF'].nunique()],
    ['Ordens REF_BACEN + NU_ORDEM distintas',dados['_CHAVE'].nunique()],
    ['ID_GLEBA distintos (identificadores de registros)',dados['_ID_REGISTRO'].nunique() if COL['id_registro'] else pd.NA],
    ['id_gleba2 distintos (geometrias identificadas)',dados['_GEOMETRIA'].nunique() if COL['geometria'] else pd.NA],
    ['Linhas sem referência BACEN',dados['_REF'].isna().sum()],
    ['Linhas sem chave completa de ordem',dados['_CHAVE'].isna().sum()],
    ['Ordens elegíveis para somas financeiras',len(ordens_validas)],
    ['Ordens não elegíveis para somas financeiras',len(ordens)-len(ordens_validas)],
    ['Soma de VL_PARC_CREDITO das ordens elegíveis (R$)',ordens_validas['VALOR_ORDEM_R'].sum(min_count=1)],
], columns=['INDICADOR','RESULTADO'])
salvar_tabela(q,'unidades_e_cobertura')
qualidade = base.apply(lambda s: texto(s).isna().sum()).rename('AUSENTES').reset_index().rename(columns={'index':'CAMPO'})
salvar_tabela(qualidade,'ausencias_por_campo',True)
salvar_tabela(dados.groupby('_FONTE_ANO').size().rename('REGISTROS').reset_index(),'origem_do_ano',True)
if len(ordens_validas) != len(ordens):
    avisar(f'{len(ordens)-len(ordens_validas):,} ordens ficaram fora das somas financeiras; veja a auditoria.')
if dados['_CHAVE'].isna().any():
    avisar('Há linhas sem chave de ordem. Permanecem nas contagens territoriais, mas não nas somas por ordem.')
display(q)


### 7. Tabelas por ano, UF, produtor, finalidade e emitente
Uma operação pode aparecer em mais de uma categoria/UF, conforme suas ordens. Assim, a soma
das barras de operações pode superar o total de operações distintas. A tabela de cobertura registra
isso explicitamente. A contagem de registros territoriais preserva todas as linhas.


In [ ]:
TABELAS, ADITIVIDADE = {}, []
for nome, (campo, rotulo) in DIMENSOES.items():
    # Finalidade ou categoria totalmente ausente: registra pendência e evita gráfico vazio.
    if nome=='finalidade' and dados[campo].eq('Não identificada').all():
        avisar('Gráficos de finalidade não gerados: falta recuperar a classificação.')
        continue
    if nome=='categoria_emitente' and not COL['emitente']:
        continue
    if nome=='categoria_produtor' and not COL['categoria']:
        continue
    if nome=='uf' and not COL['uf']:
        continue
    g = dados.groupby(campo, dropna=False)
    t = g.agg(REGISTROS_TERRITORIAIS=('_REF','size'), OPERACOES_DISTINTAS=('_REF','nunique'),
              ORDENS_DISTINTAS=('_CHAVE','nunique'), GEOMETRIAS_ID_GLEBA2=('_GEOMETRIA','nunique'))
    if not COL['geometria']:
        t['GEOMETRIAS_ID_GLEBA2'] = pd.NA
    f = ordens_validas.groupby(campo).agg(ORDENS_FINANCEIRAS=('VALOR_ORDEM_R','size'),
                  VALOR_REGISTRADO_R=('VALOR_ORDEM_R','sum'))
    t = t.join(f, how='outer')
    for c in ['REGISTROS_TERRITORIAIS','OPERACOES_DISTINTAS','ORDENS_DISTINTAS','ORDENS_FINANCEIRAS']:
        t[c] = t[c].fillna(0).astype(int)
    t['VALOR_REGISTRADO_R'] = t['VALOR_REGISTRADO_R'].fillna(0)
    t = t.reset_index().rename(columns={campo:'GRUPO'})
    if nome=='ano':
        faltantes = [str(a) for a in ANOS if str(a) not in set(t['GRUPO'])]
        if faltantes:
            t = pd.concat([t, pd.DataFrame([dict(GRUPO=a, **{c:0 for c in t.columns if c!='GRUPO'}) for a in faltantes])], ignore_index=True)
        t = t.sort_values('GRUPO')
    else:
        t = t.sort_values('REGISTROS_TERRITORIAIS', ascending=False)
    TABELAS[nome] = t
    salvar_tabela(t, 'resumo_por_'+nome)
    for col, chave in [('OPERACOES_DISTINTAS','_REF'),('ORDENS_DISTINTAS','_CHAVE')]:
        ADITIVIDADE.append({'DIMENSAO':nome,'INDICADOR':col,'SOMA_GRUPOS':int(t[col].sum()),
                            'TOTAL_DISTINTO':int(dados[chave].nunique()),
                            'IDENTIFICADORES_EM_MAIS_DE_UM_GRUPO':int((dados.groupby(chave)[campo].nunique()>1).sum())})
    assert int(t['REGISTROS_TERRITORIAIS'].sum()) == len(dados)
    if len(ordens_validas):
        assert np.isclose(float(t['VALOR_REGISTRADO_R'].sum()), float(ordens_validas['VALOR_ORDEM_R'].sum()))
    print(rotulo)
    display(t)
salvar_tabela(pd.DataFrame(ADITIVIDADE),'aditividade_das_contagens', True)
if COL['ano_territorial']:
    t_territorial = dados.groupby('_ANO_TERRITORIAL').agg(REGISTROS_TERRITORIAIS=('_REF','size'),
                          OPERACOES_DISTINTAS=('_REF','nunique'), ORDENS_DISTINTAS=('_CHAVE','nunique')).reset_index()
    salvar_tabela(t_territorial,'resumo_por_ano_do_arquivo_territorial')


### 8. Gráficos de barras e colunas
Para cada dimensão disponível, são gerados gráficos separados de registros territoriais, operações,
ordens e valores registrados. A unidade aparece no eixo vertical/horizontal e no inventário de figuras.
Os valores monetários são nominais, sem correção pela inflação.


In [ ]:
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.spines.top':False,
                     'axes.spines.right':False,'axes.titleweight':'bold','savefig.facecolor':'white'})
COR = '#216A78'
def br(x, casas=0):
    return f'{x:,.{casas}f}'.replace(',', 'X').replace('.', ',').replace('X','.')

def salvar_figura(fig, nome, titulo, unidade, nota):
    for ext in ['png','pdf']:
        fig.savefig(SAIDA/'graficos'/f'{nome}.{ext}', dpi=300, bbox_inches='tight')
    INVENTARIO.append({'ARQUIVO':nome+'.png','TITULO':titulo,'UNIDADE':unidade,'NOTA':nota})
    if MOSTRAR_GRAFICOS:
        plt.show()
    plt.close(fig)

def barras(t, coluna, nome, titulo, unidade, nota, vertical=False, dinheiro=False):
    x = t['GRUPO'].astype(str).tolist()
    y = t[coluna].astype(float).to_numpy()
    if dinheiro:
        maior = max(y.max(), 0) if len(y) else 0
        divisor, unidade = ((1e9, 'R$ bilhões (nominais)') if maior >= 1e9 else
                            (1e6, 'R$ milhões (nominais)') if maior >= 1e6 else (1, 'R$ (nominais)'))
        y = y/divisor
    def etiqueta(v):
        return '<0,01' if dinheiro and 0 < v < .005 else br(v, 2 if dinheiro else 0)
    fig, ax = plt.subplots(figsize=(10, 5.7 if vertical else max(5.7, 0.38*len(x)+2.4)))
    if vertical:
        bars = ax.bar(x, y, color=COR, width=.65)
        ax.set_ylabel(unidade)
        ax.yaxis.set_major_formatter(FuncFormatter(lambda z,p:br(z, 2 if dinheiro else 0)))
        ax.margins(y=.20)
        ax.bar_label(bars, labels=[etiqueta(v) for v in y], padding=4, fontsize=9)
        ax.grid(axis='y', alpha=.18)
        ax.tick_params(axis='x', labelsize=9)
    else:
        labels = [textwrap.fill(v, 33) for v in x]
        bars = ax.barh(labels, y, color=COR, height=.68)
        ax.invert_yaxis()
        ax.set_xlabel(unidade)
        ax.xaxis.set_major_formatter(FuncFormatter(lambda z,p:br(z, 2 if dinheiro else 0)))
        ax.margins(x=.24)
        ax.bar_label(bars, labels=[etiqueta(v) for v in y], padding=4, fontsize=9)
        ax.grid(axis='x', alpha=.18)
    ax.set_axisbelow(True)
    ax.set_title(textwrap.fill(titulo,78), loc='left', pad=15)
    rodape = 'Fonte: elaboração a partir da Base04 SICOR/Cerrado. '+nota
    fig.text(.01,.015,textwrap.fill(rodape,112),fontsize=8,va='bottom',color='#444444')
    fig.tight_layout(rect=(0,.13,1,1))
    salvar_figura(fig,nome,titulo,unidade,nota)

for i,(nome,t) in enumerate(TABELAS.items(),1):
    r = DIMENSOES[nome][1].lower()
    for j,(col,tipo,unidade) in enumerate([
        ('REGISTROS_TERRITORIAIS','Registros territoriais','Número de registros territoriais'),
        ('OPERACOES_DISTINTAS','Operações distintas','Número de REF_BACEN distintos'),
        ('ORDENS_DISTINTAS','Ordens distintas','Número de pares REF_BACEN + NU_ORDEM'),
        ('VALOR_REGISTRADO_R','Valores registrados por ordem','R$ bilhões (nominais)')],1):
        dinheiro = col=='VALOR_REGISTRADO_R'
        if dinheiro and ordens_validas.empty:
            continue
        nota = ('VL_PARC_CREDITO contado uma vez por ordem elegível; somente ordens vinculadas à base territorial. Não comprova desembolso.'
                if dinheiro else 'Contagens por grupo; uma operação ou ordem pode participar de mais de um grupo.')
        barras(t,col,f'{i:02d}_{j}_{nome}_{col.lower()}',f'{tipo} por {r} — Cerrado, 2021–2025',
               unidade,nota,vertical=nome=='ano',dinheiro=dinheiro)


### 9. Histogramas e estatísticas dos valores
São geradas três versões: distribuição completa, recorte visual até P99 e distribuição de
`log10(valor em R$)` para valores positivos. O recorte P99 não altera a base nem as somas.
Quantidade e montante acima do limite são registrados. Zeros permanecem nas versões lineares.


In [ ]:
v = ordens_validas['VALOR_ORDEM_R'].dropna().astype(float)
if v.empty:
    avisar('Sem valores elegíveis: histogramas não gerados.')
else:
    p99 = float(v.quantile(.99))
    estatisticas = pd.DataFrame([{
        'N_ORDENS':len(v), 'SOMA_R':v.sum(), 'MEDIA_R':v.mean(), 'MEDIANA_R':v.median(),
        'MIN_R':v.min(), 'P25_R':v.quantile(.25), 'P75_R':v.quantile(.75),
        'P99_R':p99,'MAX_R':v.max(),'DESVIO_PADRAO_R':v.std(),
        'N_ZERO':int(v.eq(0).sum()),'N_ACIMA_P99':int(v.gt(p99).sum()),
        'SOMA_ACIMA_P99_R':v[v.gt(p99)].sum(),
        'PERCENTUAL_VALOR_ACIMA_P99':100*v[v.gt(p99)].sum()/v.sum() if v.sum()>0 else np.nan,
    }])
    salvar_tabela(estatisticas,'estatisticas_valores_por_ordem')
    display(estatisticas.T)
    series = [('21_histograma_completo',v/1e6,'Valor registrado por ordem (R$ milhões)',
               'Distribuição completa dos valores registrados por ordem',f'n = {br(len(v))} ordens elegíveis.'),
              ('22_histograma_ate_p99',v[v.le(p99)]/1e6,'Valor registrado por ordem (R$ milhões)',
               'Distribuição dos valores registrados por ordem — até P99',
               f'P99 = R$ {br(p99,2)}; {br(int(v.gt(p99).sum()))} ordens acima do limite não exibidas.'),
              ('23_histograma_log10',np.log10(v[v.gt(0)]),'log10(valor registrado por ordem, em R$)',
               'Distribuição de log10 dos valores registrados por ordem',
               f'Somente valores positivos; {br(int(v.eq(0).sum()))} ordens com valor zero não exibidas.')]
    for nome, valores, eixo, titulo, nota in series:
        if len(valores)==0:
            continue
        bins = min(60, max(8,int(np.sqrt(len(valores)))))
        fig,ax = plt.subplots(figsize=(10,5.9))
        freq, edges, _ = ax.hist(valores,bins=bins,color=COR,edgecolor='white',linewidth=.5)
        ax.set_title(titulo+'\nCerrado, 2021–2025',loc='left',pad=15)
        ax.set_xlabel(eixo)
        ax.set_ylabel('Número de ordens')
        ax.yaxis.set_major_formatter(FuncFormatter(lambda z,p:br(z)))
        ax.xaxis.set_major_formatter(FuncFormatter(lambda z,p:br(z,2)))
        ax.grid(axis='y',alpha=.18)
        fig.text(.01,.02,textwrap.fill('Fonte: Base04 SICOR/Cerrado. '+nota+' VL_PARC_CREDITO contado uma vez por ordem; não representa desembolso confirmado.',115),fontsize=8)
        fig.tight_layout(rect=(0,.12,1,1))
        salvar_figura(fig,nome,titulo,'Ordens/destinações',nota)
        salvar_tabela(pd.DataFrame({'LIMITE_INFERIOR':edges[:-1], 'LIMITE_SUPERIOR':edges[1:],
                     'FREQUENCIA':freq.astype(int),'ESCALA':eixo}),nome+'_classes')


### 10. Resposta à dúvida sobre execução financeira
Esta célula informa quais campos foram encontrados; **não classifica automaticamente** contratos
como pagos, cancelados ou financiados. `DT_VENCIMENTO` ou `VL_PARC_CREDITO` não comprovam pagamento.
A ausência de uma liberação em um arquivo incompleto também não prova que não houve desembolso.

Para responder empiricamente à segunda parte do comentário, será necessário integrar as tabelas
de liberação e de saldos/situação, respeitando chaves, datas e cobertura. Os códigos de situação
dependem de sua tabela de domínio. Este notebook não baixa nem integra essas tabelas financeiras.


In [ ]:
perguntas = [
    ('Há informação de recursos liberados?', ['VL_LIBERADO','DT_LIBERACAO'],
     'Valores e datas de liberação exigem conferência de cobertura e vínculos.'),
    ('Há informação de situação da operação?', ['CD_SITUACAO_OPERACAO'],
     'Interpretar com a tabela de domínio e o mês/ano de referência dos saldos.'),
    ('Há informação de desclassificação?', ['DT_DESC','VL_DESC','CD_MOTIVO_DESC'],
     'Desclassificação, isoladamente, não comprova cancelamento antes de qualquer liberação.'),
]
diagnostico = []
for pergunta,nomes,limite in perguntas:
    presentes = [localizar(cabecalho,[n]) for n in nomes]
    presentes = [c for c in presentes if c]
    diagnostico.append({'PERGUNTA':pergunta,'CAMPOS_ENCONTRADOS':', '.join(presentes) or 'Nenhum',
                       'CONCLUSAO':'Não verificado neste notebook','LIMITACAO':limite})
diagnostico = pd.DataFrame(diagnostico)
salvar_tabela(diagnostico,'diagnostico_execucao_financeira',True)
display(diagnostico)
print('Os gráficos mostram registros administrativos e valores registrados; não comprovam liberação, quitação ou cancelamento.')


### 11. Relatório, inventário e ZIP
Consulte primeiro `LEIA_RESULTADOS.md`, depois as figuras. O inventário fornece títulos e unidades
para as legendas da dissertação. O arquivo ZIP inclui as auditorias; ele pode conter referências
BACEN e deve ser compartilhado conforme a finalidade da pesquisa.


In [ ]:
assert len(dados) == N_ORIGINAL
salvar_tabela(pd.DataFrame(INVENTARIO),'inventario_figuras')
manifesto = {'execucao_utc':EXECUCAO,'fontes':FONTES,'colunas':COL,
            'linhas_preservadas':len(dados),'anos':ANOS,'formato_numerico':FORMATO_NUMERICO,
            'regra_ano':'DT_EMISSAO; na ausência, campo de ano identificado',
            'regra_financeira':'Um valor consistente por REF_BACEN + NU_ORDEM; somente ordens territorialmente vinculadas',
            'versoes':{'pandas':pd.__version__,'numpy':np.__version__},'avisos':AVISOS}
(SAIDA/'auditoria/manifesto_execucao.json').write_text(json.dumps(manifesto,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
resumo = f'''# Resultados — seção 5.2.1

Execução: {EXECUCAO}

Base: {ARQUIVO_BASE}

Foram preservados {br(len(dados))} registros territoriais. Foram identificadas
{br(dados['_REF'].nunique())} operações e {br(dados['_CHAVE'].nunique())} ordens distintas.
O campo ID_GLEBA é identificador de registro; id_gleba2 identifica geometrias repetidas.
Não se assume que nenhum desses totais seja o número de imóveis rurais.

## Figuras para a dissertação
Use os gráficos de operações por ano, UF e categoria de produtor, e finalidade quando disponível.
Para a estrutura territorial, use os gráficos de registros territoriais, com essa unidade na legenda.
Use um dos histogramas por ordem: o completo, o até P99 ou o de log10. Explique o recorte/transformação.
As versões PNG têm 300 dpi; os PDFs preservam elementos vetoriais.
Veja tabelas/inventario_figuras.csv para nomes e títulos.

## Como interpretar
- Quantidades por categoria são barras, não histogramas.
- Uma operação pode participar de mais de um grupo. Veja auditoria/aditividade_das_contagens.csv.
- TIPO_PRODUTOR_ANALISE é a classificação existente na Base04, sem nova inferência de enquadramento.
- UF_OPERACAO é uma UF administrativa; não foi realizado novo cruzamento espacial.
- O ano prioriza DT_EMISSAO. Veja auditoria/origem_do_ano.csv para uso de ano auxiliar.
- Os valores são nominais, registrados, contados uma vez por ordem com vínculo territorial.
- {br(len(ordens_validas))} ordens são elegíveis às somas; {br(len(ordens)-len(ordens_validas))} foram separadas para auditoria.
- Os totais financeiros podem ser parciais tanto por qualidade do valor quanto por cobertura territorial.
- Nenhum pagamento, cancelamento ou desembolso foi confirmado por este processamento.

## Relação com a Tabela 3
Não some registros, ordens, operações e geometrias: são unidades distintas.
As ordens sem correspondência territorial não são recuperáveis apenas a partir desta Base04.
Os números 401.294, 374.949, 26.345 e 454.817 do texto anterior não foram impostos ao resultado.
Verifique a regra de contagem usada originalmente para 454.817 antes de equipará-la a um novo indicador.

## Pendências e avisos
'''
resumo += '\n'.join('- '+a for a in AVISOS) if AVISOS else 'Nenhuma pendência automática registrada. A execução financeira permanece não verificada.'
(SAIDA/'LEIA_RESULTADOS.md').write_text(resumo,encoding='utf-8')
zip_path = shutil.make_archive(str(SAIDA), 'zip', root_dir=SAIDA)
print('Concluído:', len(INVENTARIO), 'figuras em PNG e PDF.')
print('Pasta:', SAIDA)
print('ZIP:', zip_path)
if BAIXAR_ZIP_AO_FINAL:
    from google.colab import files
    files.download(zip_path)
